# Direct Demand Model for Corridor Ridership — Economy & 1st Class

This notebook builds a **direct demand model** for quarterly ridership on a transit
corridor, separately for **Economy Class** and **1st Class**, then investigates the
**role of lagged demand** (habit / partial-adjustment effects) in each.

Run the cells top to bottom. Each section prints the fitted model, its diagnostics,
and a plain-language interpretation generated from the actual fitted numbers (not
hardcoded), so re-running with edited data or specifications stays self-checking.

**Structure**
1. Setup
2. Load & inspect data
3. Data preparation (logs, seasonal dummies, lags)
4. Exploratory correlations
5. Static model — Economy
6. Static model — 1st Class
7. Multicollinearity check (VIF) — static
8. Dynamic model (lagged demand) — Economy
9. Dynamic model (lagged demand) — 1st Class
10. Multicollinearity check (VIF) — dynamic
11. Short-run vs long-run elasticities
12. Autocorrelation diagnostics (Durbin-Watson / Durbin's h)
13. Model comparison summary
14. Conclusions on lagged demand


## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.stattools import durbin_watson

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 30)
ALPHA = 0.05  # significance threshold used throughout

SRC_PATH = "TransitRidershipData.xlsx"  # run this notebook from the folder containing the file


## 2. Load & Inspect Data

Confirms the raw file loaded correctly: shape, column types, and summary
statistics. Check this against the source workbook before trusting anything
downstream.


In [ ]:
raw = pd.read_excel(SRC_PATH)
print(f"Loaded {raw.shape[0]} rows x {raw.shape[1]} columns")
display(raw.head())


In [ ]:
print("Column dtypes:")
print(raw.dtypes)
print()
print("Summary statistics:")
display(raw.describe().T)
print()
missing = raw.isnull().sum()
print("Missing values per column (should all be 0):")
print(missing[missing > 0] if missing.any() else "None - dataset is complete.")


## 3. Data Preparation

- Rename columns to short handles.
- Log-transform ridership and the continuous economic/price regressors → gives a
  **log-log (constant elasticity)** specification, so coefficients are directly
  interpretable as elasticities.
- `Delays` (on-time performance) is left in levels → its coefficient is a
  **semi-elasticity** (% change in ridership per one-unit change in delay count).
- Quarter dummies (`Q_2`, `Q_3`, `Q_4`, with Q1 as reference) capture seasonality.
- One-quarter lags of each class's own log-ridership are built for the dynamic
  (partial-adjustment) models in Sections 8-9. Building the lag costs the notebook
  its first observation (Year 1, Quarter 1) in every model that uses it.


In [ ]:
df = raw.sort_values(["Year", "Quarter"]).reset_index(drop=True)
df["t"] = range(1, len(df) + 1)  # chronological index, useful for plots

df = df.rename(columns={
    "Total Number of Passengers: Economy Class": "Q_econ",
    "Total Number of Passengers: 1st Class": "Q_1st",
    "Avg Fare Per KM of Economyclass": "Fare_econ",
    "Avg Fare Per KM of 1st class": "Fare_1st",
    "On Time Performances: Number of Times Schedule Delays happened": "Delays",
    "Total Employment in the Region along the Corridor": "Employment",
    "Avg Gas Price along the Corridor": "GasPrice",
    "Avg Air Fare Per KM along the Corridor": "AirFare",
})

LOG_VARS = ["Q_econ", "Q_1st", "Fare_econ", "Fare_1st", "Employment", "GasPrice", "AirFare"]
for col in LOG_VARS:
    df[f"ln_{col}"] = np.log(df[col])

df["Quarter"] = df["Quarter"].astype(int)
df = pd.get_dummies(df, columns=["Quarter"], prefix="Q", drop_first=True)
for c in ["Q_2", "Q_3", "Q_4"]:
    df[c] = df[c].astype(int)

# one-quarter lags of own (log) ridership, built BEFORE dropping any rows
df["ln_Q_econ_lag1"] = df["ln_Q_econ"].shift(1)
df["ln_Q_1st_lag1"] = df["ln_Q_1st"].shift(1)

df_dyn = df.dropna(subset=["ln_Q_econ_lag1", "ln_Q_1st_lag1"]).copy()

print(f"Static-model sample size: {len(df)} quarters")
print(f"Dynamic-model sample size: {len(df_dyn)} quarters (first quarter dropped for the lag)")
print()
display(df[["Year", "t"] + [c for c in df.columns if c.startswith("Q_")] +
           ["ln_Q_econ", "ln_Q_econ_lag1", "ln_Q_1st", "ln_Q_1st_lag1"]].head(6))


## 4. Exploratory Correlations

Before fitting anything, look at how correlated the candidate regressors are.
Several of these (fares, employment, air fare, gas price) plausibly trend
together over the 9 years — that shows up here as high pairwise correlation and
foreshadows the multicollinearity (VIF) results in Sections 7 and 10.


In [ ]:
corr_vars = ["ln_Q_econ", "ln_Q_1st", "ln_Fare_econ", "ln_Fare_1st",
             "ln_AirFare", "ln_GasPrice", "ln_Employment", "Delays"]
corr = df[corr_vars].corr().round(2)
display(corr)

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(corr, vmin=-1, vmax=1, cmap="RdBu_r")
ax.set_xticks(range(len(corr_vars))); ax.set_xticklabels(corr_vars, rotation=45, ha="right")
ax.set_yticks(range(len(corr_vars))); ax.set_yticklabels(corr_vars)
for i in range(len(corr_vars)):
    for j in range(len(corr_vars)):
        ax.text(j, i, f"{corr.values[i, j]:.2f}", ha="center", va="center", fontsize=8)
fig.colorbar(im, label="Pearson correlation")
ax.set_title("Correlation matrix — regressors and dependent variables")
plt.tight_layout()
plt.show()

high = [(corr_vars[i], corr_vars[j], corr.values[i, j])
        for i in range(len(corr_vars)) for j in range(i + 1, len(corr_vars))
        if abs(corr.values[i, j]) > 0.7]
print("Pairs with |correlation| > 0.7 (flag for potential multicollinearity):")
for a, b, v in high:
    print(f"  {a} <-> {b}: {v:.2f}")
if not high:
    print("  None found.")


## 5. Static Model — Economy Class

Direct demand model, no lag:

`ln(Q_econ) = b0 + b1*ln(Fare_econ) + b2*ln(Fare_1st) + b3*ln(AirFare) + b4*ln(GasPrice) + b5*ln(Employment) + b6*Delays + Q2 + Q3 + Q4`


In [ ]:
RHS_STATIC = ("ln_Fare_econ + ln_Fare_1st + ln_AirFare + ln_GasPrice + "
              "ln_Employment + Delays + Q_2 + Q_3 + Q_4")

VAR_META = {
    "ln_Fare_econ":  ("elasticity", "Economy own fare"),
    "ln_Fare_1st":   ("elasticity", "1st Class fare (cross-price)"),
    "ln_AirFare":    ("elasticity", "Competing air fare"),
    "ln_GasPrice":   ("elasticity", "Gas price (driving cost proxy)"),
    "ln_Employment": ("elasticity", "Regional employment (economic activity)"),
    "Delays":        ("semi-elasticity", "Schedule delay count (service quality)"),
    "Q_2": ("dummy", "Quarter 2 vs Quarter 1"),
    "Q_3": ("dummy", "Quarter 3 vs Quarter 1"),
    "Q_4": ("dummy", "Quarter 4 vs Quarter 1"),
}

def interpret_model(model, meta=VAR_META, lag_name=None, alpha=ALPHA):
    """Print a plain-language read of each coefficient, computed from the fitted model."""
    print(f"R-squared: {model.rsquared:.3f}   Adj. R-squared: {model.rsquared_adj:.3f}   "
          f"N = {int(model.nobs)}   F p-value: {model.f_pvalue:.2e}")
    print("-" * 90)
    for var in model.params.index:
        if var == "Intercept":
            continue
        coef, p = model.params[var], model.pvalues[var]
        sig = "significant" if p < alpha else "not significant"
        star = "*" if p < alpha else " "
        if lag_name is not None and var == lag_name:
            print(f"{star} {var:16s} gamma={coef:+.4f}  p={p:.3f}  ({sig}) "
                  f"-> speed of adjustment toward equilibrium demand")
            continue
        kind, label = meta.get(var, ("elasticity", var))
        if kind == "elasticity":
            print(f"{star} {var:16s} coef={coef:+.4f}  p={p:.3f}  ({sig}) "
                  f"-> {label}: 1% change -> {coef:+.3f}% change in ridership")
        elif kind == "semi-elasticity":
            print(f"{star} {var:16s} coef={coef:+.5f}  p={p:.3f}  ({sig}) "
                  f"-> {label}: +1 unit -> {coef*100:+.3f}% change in ridership")
        else:  # dummy
            pct = (np.exp(coef) - 1) * 100
            print(f"{star} {var:16s} coef={coef:+.4f}  p={p:.3f}  ({sig}) "
                  f"-> {label}: {pct:+.1f}% ridership relative to Q1, holding other factors fixed")
    print("-" * 90)
    print("(* = significant at the", f"{int(alpha*100)}%", "level)")

m_econ_static = smf.ols(f"ln_Q_econ ~ {RHS_STATIC}", data=df).fit()
print(m_econ_static.summary())


In [ ]:
print("INTERPRETATION - Economy, static model")
interpret_model(m_econ_static)


## 6. Static Model — 1st Class

Same specification, dependent variable is `ln(Q_1st)`. Fitted independently of
the Economy model since the two segments may respond differently to the same
drivers.


In [ ]:
m_1st_static = smf.ols(f"ln_Q_1st ~ {RHS_STATIC}", data=df).fit()
print(m_1st_static.summary())


In [ ]:
print("INTERPRETATION - 1st Class, static model")
interpret_model(m_1st_static)


## 7. Multicollinearity Check (VIF) — Static Regressors

Variance Inflation Factors for the shared regressor set. **Rule of thumb: VIF >
10 signals problematic multicollinearity** — coefficients on those variables may
have inflated standard errors (large p-values) even if the true relationship is
real. This is the likely explanation if Sections 5-6 showed economically sensible
but statistically insignificant coefficients.


In [ ]:
def vif_table(data, columns):
    X = data[columns].copy()
    X.insert(0, "const", 1.0)
    out = pd.DataFrame({
        "variable": X.columns,
        "VIF": [variance_inflation_factor(X.values, i) for i in range(X.shape[1])],
    })
    return out[out["variable"] != "const"].reset_index(drop=True)

static_cols = ["ln_Fare_econ", "ln_Fare_1st", "ln_AirFare", "ln_GasPrice",
               "ln_Employment", "Delays", "Q_2", "Q_3", "Q_4"]
vif_static = vif_table(df, static_cols)
display(vif_static)

flagged = vif_static[vif_static["VIF"] > 10]
if len(flagged):
    print("Variables with VIF > 10 (multicollinearity concern):")
    print(flagged.to_string(index=False))
else:
    print("No variable exceeds VIF = 10.")


## 8. Dynamic Model (Lagged Demand) — Economy Class

Adds the one-quarter-lagged own log-ridership to test for habit persistence /
partial adjustment:

`ln(Q_econ,t) = b0 + gamma*ln(Q_econ,t-1) + b1*ln(Fare_econ) + ... + Q2 + Q3 + Q4`

**gamma** is the adjustment-speed parameter: close to 0 means demand adjusts to
its new equilibrium almost immediately each quarter; close to 1 means adjustment
is slow (strong habit/inertia). If significant, the **long-run elasticity** of
each driver is short-run coefficient / (1 - gamma) — see Section 11.


In [ ]:
m_econ_dyn = smf.ols(f"ln_Q_econ ~ ln_Q_econ_lag1 + {RHS_STATIC}", data=df_dyn).fit()
print(m_econ_dyn.summary())


In [ ]:
print("INTERPRETATION - Economy, dynamic model")
interpret_model(m_econ_dyn, lag_name="ln_Q_econ_lag1")

gamma = m_econ_dyn.params["ln_Q_econ_lag1"]
p_gamma = m_econ_dyn.pvalues["ln_Q_econ_lag1"]
print()
if p_gamma < ALPHA:
    print(f"=> Lagged demand IS statistically significant for Economy (gamma={gamma:.3f}, p={p_gamma:.3f}).")
else:
    print(f"=> Lagged demand is NOT statistically significant for Economy (gamma={gamma:.3f}, p={p_gamma:.3f}).")
print(f"   Adj. R-squared static={m_econ_static.rsquared_adj:.3f} vs dynamic={m_econ_dyn.rsquared_adj:.3f}")


## 9. Dynamic Model (Lagged Demand) — 1st Class

In [ ]:
m_1st_dyn = smf.ols(f"ln_Q_1st ~ ln_Q_1st_lag1 + {RHS_STATIC}", data=df_dyn).fit()
print(m_1st_dyn.summary())


In [ ]:
print("INTERPRETATION - 1st Class, dynamic model")
interpret_model(m_1st_dyn, lag_name="ln_Q_1st_lag1")

gamma = m_1st_dyn.params["ln_Q_1st_lag1"]
p_gamma = m_1st_dyn.pvalues["ln_Q_1st_lag1"]
print()
if p_gamma < ALPHA:
    print(f"=> Lagged demand IS statistically significant for 1st Class (gamma={gamma:.3f}, p={p_gamma:.3f}).")
else:
    print(f"=> Lagged demand is NOT statistically significant for 1st Class (gamma={gamma:.3f}, p={p_gamma:.3f}).")
print(f"   Adj. R-squared static={m_1st_static.rsquared_adj:.3f} vs dynamic={m_1st_dyn.rsquared_adj:.3f}")


## 10. Multicollinearity Check (VIF) — Dynamic Regressors

Adding the lagged dependent variable typically raises VIFs further, since
ridership is autocorrelated with the same trending drivers.


In [ ]:
dyn_cols_econ = ["ln_Q_econ_lag1"] + static_cols
dyn_cols_1st = ["ln_Q_1st_lag1"] + static_cols

print("Economy dynamic model VIFs:")
display(vif_table(df_dyn, dyn_cols_econ))

print("1st Class dynamic model VIFs:")
display(vif_table(df_dyn, dyn_cols_1st))


## 11. Short-Run vs Long-Run Elasticities

For the dynamic models, the short-run elasticity is the fitted coefficient; the
long-run elasticity — the full effect once ridership fully adjusts — is
`coefficient / (1 - gamma)`. The gap between the two widens as gamma (habit
persistence) grows.


In [ ]:
def sr_lr_table(model, lag_name):
    gamma = model.params[lag_name]
    rows = []
    for var in model.params.index:
        if var in ("Intercept", lag_name):
            continue
        sr = model.params[var]
        lr = sr / (1 - gamma) if gamma != 1 else np.nan
        rows.append((var, sr, lr, model.pvalues[var]))
    out = pd.DataFrame(rows, columns=["variable", "short_run_elasticity", "long_run_elasticity", "p_value"])
    out.attrs["gamma"] = gamma
    return out

econ_elast = sr_lr_table(m_econ_dyn, "ln_Q_econ_lag1")
print(f"Economy: gamma = {econ_elast.attrs['gamma']:.4f}")
display(econ_elast.round(4))

first_elast = sr_lr_table(m_1st_dyn, "ln_Q_1st_lag1")
print(f"1st Class: gamma = {first_elast.attrs['gamma']:.4f}")
display(first_elast.round(4))


## 12. Autocorrelation Diagnostics (Durbin-Watson / Durbin's h)

Durbin-Watson (DW) is valid for the static models (values near 2 = no
autocorrelation; well below 2 = positive autocorrelation). **DW is biased
toward 2 once a lagged dependent variable is included**, so for the dynamic
models Durbin's h-test is the correct diagnostic. Durbin's h can be undefined
(`None` below) in small samples when `n * se(gamma)^2 >= 1` — if that happens,
DW is reported only as a rough guide, not a formal test.


In [ ]:
def durbins_h(model, lag_name):
    n = int(model.nobs)
    dw = durbin_watson(model.resid)
    gamma_hat = model.params[lag_name]
    se_gamma = model.bse[lag_name]
    denom = 1 - n * (se_gamma ** 2)
    h = (1 - dw / 2) * np.sqrt(n / denom) if denom > 0 else None
    return dw, h

print("Static models (Durbin-Watson valid directly):")
print(f"  Economy static  : DW = {durbin_watson(m_econ_static.resid):.3f}")
print(f"  1st Class static: DW = {durbin_watson(m_1st_static.resid):.3f}")
print("  (Rule of thumb: ~2 = no autocorrelation; < 1.5 = positive autocorrelation warning)")
print()

print("Dynamic models (Durbin's h is the correct test; DW shown for reference only):")
dw_e, h_e = durbins_h(m_econ_dyn, "ln_Q_econ_lag1")
dw_f, h_f = durbins_h(m_1st_dyn, "ln_Q_1st_lag1")
print(f"  Economy dynamic  : DW = {dw_e:.3f}, Durbin's h = {h_e if h_e is None else round(h_e, 3)}")
print(f"  1st Class dynamic: DW = {dw_f:.3f}, Durbin's h = {h_f if h_f is None else round(h_f, 3)}")
if h_e is None or h_f is None:
    print("  Note: h undefined for at least one model (small-sample condition n*se(gamma)^2 >= 1);"
          " rely on the DW value directionally and note this limitation in your report.")


## 13. Model Comparison Summary

Side-by-side static vs dynamic fit statistics for both classes, to judge whether
adding lagged demand actually earns its place in the model.


In [ ]:
summary_rows = []
for cls, static_m, dyn_m, lag_name in [
    ("Economy", m_econ_static, m_econ_dyn, "ln_Q_econ_lag1"),
    ("1st Class", m_1st_static, m_1st_dyn, "ln_Q_1st_lag1"),
]:
    summary_rows.append({
        "class": cls, "model": "static",
        "adj_R2": static_m.rsquared_adj, "AIC": static_m.aic, "BIC": static_m.bic,
        "gamma": np.nan, "gamma_p": np.nan,
    })
    summary_rows.append({
        "class": cls, "model": "dynamic",
        "adj_R2": dyn_m.rsquared_adj, "AIC": dyn_m.aic, "BIC": dyn_m.bic,
        "gamma": dyn_m.params[lag_name], "gamma_p": dyn_m.pvalues[lag_name],
    })

comparison = pd.DataFrame(summary_rows)
display(comparison.round(4))

print()
print("Lower AIC/BIC and higher adj. R-squared favor that specification.")
for cls in ["Economy", "1st Class"]:
    sub = comparison[comparison["class"] == cls].set_index("model")
    better = "dynamic" if sub.loc["dynamic", "AIC"] < sub.loc["static", "AIC"] else "static"
    print(f"  {cls}: AIC favors the {better} model.")


## 14. Conclusions on Lagged Demand

Read the printed gamma significance flags from Sections 8-9 and the AIC/BIC
comparison from Section 13 together:

- If **gamma is significant** and AIC/BIC favor the dynamic model: lagged demand
  (habit persistence / partial adjustment) is a real feature of that segment's
  ridership — report short-run vs long-run elasticities from Section 11 as your
  headline result for that class.
- If **gamma is not significant**: the static model is adequate for that class in
  this sample: quarter-to-quarter ridership responds essentially fully within
  the same quarter, or the sample (33-35 observations) and the multicollinearity
  flagged in Sections 7 and 10 aren't powerful enough to detect a lagged effect
  even if one exists. Say so explicitly rather than overclaiming — don't inflate
  a p=0.13 into "there is a lagged demand effect".

Re-run this notebook after any change to the data or specification (e.g. trimming
collinear regressors, swapping `Employment` for a year trend) — every printed
number above is recomputed live, not hardcoded.
